In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, make_scorer
from sklearn.feature_selection import SelectFromModel
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def comprehensive_regularization_analysis():
    """
    Implements comparative analysis of three regularization approaches:
    1. Regularização Padrão (Lasso, Ridge, Elastic Net)
    2. Group-aware Feature Engineering  
    3. Regularização com Feature Groups Manual
    
    Inclui nested cross-validation e intervalos de confiança de 95%
    """
    
    # Load training data
    train_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL1TRAIN.csv'
    df = pd.read_csv(train_path)
    
    print("="*80)
    print("COMPARATIVE REGULARIZATION ANALYSIS - MODELO 1")
    print("="*80)
    
    # Prepare data
    # Target: obesidade (binário)
    y = df['obeso'].copy()
    
    # Features: excluir id_anon e todas as variables target
    target_cols = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
    feature_cols = [col for col in df.columns if col not in target_cols + ['id_anon']]
    X = df[feature_cols].copy()
    
    print(f"Dataset de treinamento:")
    print(f"  - Observations: {len(df):,}")
    print(f"  - Features: {len(feature_cols)}")
    print(f"  - Obese: {y.sum()} ({y.mean()*100:.1f}%)")
    print(f"  - Não-obesos: {(1-y).sum()} ({(1-y.mean())*100:.1f}%)")
    
    # Define feature groups for analysis
    feature_groups = {
        'regiao': ['regiao_Centro-Oeste', 'regiao_Nordeste', 'regiao_Norte', 'regiao_Sudeste', 'regiao_Sul'],
        'cor_raca': ['cor_Branca', 'cor_Outros', 'cor_Parda (mulata, cabocla, cafuza, mameluca ou mestiça)', 'cor_Preta'],
        'tipo_parto': ['parto_Cesariana agendada (eletiva)', 'parto_Cesariana de urgência (Não agendada)', 'parto_Normal'],
        'demograficas': ['sexo_masculino', 'idade_crianca', 'idade_materna_ao_nascimento'],
        'perinatais': ['semanas_gravidez', 'peso_nascimento', 'altura_nascimento', 'tipo_de_parto'],
        'socioeconomicas': ['sabe_ler', 'nivel_educacional_mae', 'recebe_beneficio', 'vd_ien_escore'],
        'outras': ['chupeta_usou', 'sindrome_nao', 'teve_exposicao_escola', 'k05_prenatal_consultas']
    }
    
    print(f"\nFeature groups identified:")
    for grupo, features in feature_groups.items():
        print(f"  - {grupo.capitalize()}: {len(features)} variables")
    
    # Cross-validation settings
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Metrics para evaluation
    scoring = {
        'auc': 'roc_auc',
        'precision': make_scorer(precision_score, zero_division=0),
        'recall': make_scorer(recall_score, zero_division=0),
        'f1': make_scorer(f1_score, zero_division=0)
    }
    
    # Parameters for grid search
    param_grids = {
        'lasso': {
            'model__C': [0.1, 1.0, 10.0, 100.0, 1000.0],  # Inverse of alpha
            'model__penalty': ['l1']
        },
        'ridge': {
            'model__C': [0.1, 1.0, 10.0, 100.0, 1000.0],  # Inverse of alpha
            'model__penalty': ['l2']
        },
        'elastic_net': {
            'model__C': [0.1, 1.0, 10.0, 100.0, 1000.0],
            'model__penalty': ['elasticnet'],
            'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
        }
    }
    
    def calculate_confidence_intervals(scores, confidence=0.95):
        """Calculates confidence intervals using t-distribution"""
        n = len(scores)
        mean_score = np.mean(scores)
        std_err = stats.sem(scores)
        h = std_err * stats.t.ppf((1 + confidence) / 2., n-1)
        return mean_score, mean_score - h, mean_score + h
    
    def evaluate_regularization_method(X_data, y_data, model, param_grid, method_name):
        """Evaluates a regularization method with nested cross-validation"""
        
        print(f"\n[SEARCH] Evaluating {method_name}...")
        
        # Pipeline with preprocessing
        pipeline = Pipeline([
            ('scaler', RobustScaler()),
            ('model', model)
        ])
        
        # Nested cross-validation
        results = []
        best_params_list = []
        selected_features_list = []
        
        for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_data, y_data)):
            X_train_fold, X_val_fold = X_data.iloc[train_idx], X_data.iloc[val_idx]
            y_train_fold, y_val_fold = y_data.iloc[train_idx], y_data.iloc[val_idx]
            
            # Grid search no inner CV
            grid_search = GridSearchCV(
                pipeline, param_grid, cv=inner_cv, 
                scoring='roc_auc', n_jobs=-1, verbose=0
            )
            
            grid_search.fit(X_train_fold, y_train_fold)
            best_params_list.append(grid_search.best_params_)
            
            # Avaliar no validation fold
            y_pred_proba = grid_search.predict_proba(X_val_fold)[:, 1]
            y_pred = grid_search.predict(X_val_fold)
            
            # Calcular métricas
            fold_results = {
                'auc': roc_auc_score(y_val_fold, y_pred_proba),
                'precision': precision_score(y_val_fold, y_pred, zero_division=0),
                'recall': recall_score(y_val_fold, y_pred, zero_division=0),
                'f1': f1_score(y_val_fold, y_pred, zero_division=0)
            }
            results.append(fold_results)
            
            # Features selecionadas (para Lasso e Elastic Net)
            if hasattr(grid_search.best_estimator_.named_steps['model'], 'coef_'):
                coefs = grid_search.best_estimator_.named_steps['model'].coef_
                # Para classification binária, coef_ pode ter shape (1, n_features) ou (n_features,)
                if coefs.ndim > 1:
                    coefs = coefs.flatten()
                selected_features = X_data.columns[coefs != 0].tolist()
                selected_features_list.append(selected_features)
        
        # Consolidate results
        metrics_summary = {}
        for metric in ['auc', 'precision', 'recall', 'f1']:
            scores = [r[metric] for r in results]
            mean_score, ci_lower, ci_upper = calculate_confidence_intervals(scores)
            metrics_summary[metric] = {
                'mean': mean_score,
                'std': np.std(scores),
                'ci_lower': ci_lower,
                'ci_upper': ci_upper,
                'scores': scores
            }
        
        return {
            'method': method_name,
            'metrics': metrics_summary,
            'best_params': best_params_list,
            'selected_features': selected_features_list,
            'n_features_selected': [len(sf) for sf in selected_features_list] if selected_features_list else None
        }
    
    # =======================================================================
    # APPROACH 1: STANDARD REGULARIZATION
    # =======================================================================
    print(f"\n" + "="*60)
    print("APPROACH 1: STANDARD REGULARIZATION")
    print("="*60)
    
    standard_results = {}
    
    # Lasso
    lasso_results = evaluate_regularization_method(
        X, y, LogisticRegression(random_state=42, max_iter=10000, penalty='l1', solver='liblinear', class_weight='balanced'), 
        param_grids['lasso'], 'Lasso'
    )
    standard_results['lasso'] = lasso_results
    
    # Ridge  
    ridge_results = evaluate_regularization_method(
        X, y, LogisticRegression(random_state=42, max_iter=10000, penalty='l2', class_weight='balanced'), 
        param_grids['ridge'], 'Ridge'
    )
    standard_results['ridge'] = ridge_results
    
    # Elastic Net
    elastic_results = evaluate_regularization_method(
        X, y, LogisticRegression(random_state=42, max_iter=10000, penalty='elasticnet', solver='saga', class_weight='balanced'), 
        param_grids['elastic_net'], 'Elastic Net'
    )
    standard_results['elastic_net'] = elastic_results
    
    # =======================================================================
    # APPROACH 2: GROUP-BASED FEATURE ENGINEERING
    # =======================================================================
    print(f"\n" + "="*60)
    print("APPROACH 2: GROUP-BASED FEATURE ENGINEERING")
    print("="*60)
    
    # Create aggregated features for categorical variables
    X_engineered = X.copy()
    
    # Region: North/Northeast vs South/Southeast/Central-West (epidemiological division)
    X_engineered['regiao_norte_nordeste'] = (X['regiao_Norte'] + X['regiao_Nordeste']).clip(0, 1)
    X_engineered = X_engineered.drop(feature_groups['regiao'], axis=1)
    
    # Cor/raça: Branca vs Não-branca (common epidemiological division)
    X_engineered['cor_nao_branca'] = 1 - X['cor_Branca']
    X_engineered = X_engineered.drop(feature_groups['cor_raca'], axis=1)
    
    # Tipo de parto: Cesariana (qualquer tipo) vs Normal
    X_engineered['parto_cesariana'] = (X['parto_Cesariana agendada (eletiva)'] + 
                                      X['parto_Cesariana de urgência (Não agendada)']).clip(0, 1)
    X_engineered = X_engineered.drop(feature_groups['tipo_parto'], axis=1)
    
    print(f"Features after engineering:")
    print(f"  - Original: {X.shape[1]} features")
    print(f"  - Engineered: {X_engineered.shape[1]} features")
    print(f"  - Reduction: {X.shape[1] - X_engineered.shape[1]} features")
    
    # Evaluate methods on engineered version
    engineered_results = {}
    
    lasso_eng_results = evaluate_regularization_method(
        X_engineered, y, LogisticRegression(random_state=42, max_iter=10000, penalty='l1', solver='liblinear', class_weight='balanced'), 
        param_grids['lasso'], 'Lasso (Engineered)'
    )
    engineered_results['lasso'] = lasso_eng_results
    
    ridge_eng_results = evaluate_regularization_method(
        X_engineered, y, LogisticRegression(random_state=42, max_iter=10000, penalty='l2', class_weight='balanced'), 
        param_grids['ridge'], 'Ridge (Engineered)'
    )
    engineered_results['ridge'] = ridge_eng_results
    
    elastic_eng_results = evaluate_regularization_method(
        X_engineered, y, LogisticRegression(random_state=42, max_iter=10000, penalty='elasticnet', solver='saga', class_weight='balanced'), 
        param_grids['elastic_net'], 'Elastic Net (Engineered)'
    )
    engineered_results['elastic_net'] = elastic_eng_results
    
    # =======================================================================
    # COMPARATIVE RESULTS
    # =======================================================================
    print(f"\n" + "="*80)
    print("COMPARATIVE RESULTS")
    print("="*80)
    
    def print_results_table(results_dict, approach_name):
        print(f"\n{approach_name.upper()}:")
        print(f"{'Método':<15} {'AUC-ROC':<25} {'Precision':<25} {'Recall':<25} {'F1-Score':<25}")
        print("-" * 115)
        
        for method_key, result in results_dict.items():
            method_name = result['method']
            auc = result['metrics']['auc']
            precision = result['metrics']['precision']
            recall = result['metrics']['recall']
            f1 = result['metrics']['f1']
            
            auc_str = f"{auc['mean']:.3f} ({auc['ci_lower']:.3f}-{auc['ci_upper']:.3f})"
            prec_str = f"{precision['mean']:.3f} ({precision['ci_lower']:.3f}-{precision['ci_upper']:.3f})"
            rec_str = f"{recall['mean']:.3f} ({recall['ci_lower']:.3f}-{recall['ci_upper']:.3f})"
            f1_str = f"{f1['mean']:.3f} ({f1['ci_lower']:.3f}-{f1['ci_upper']:.3f})"
            
            print(f"{method_name:<15} {auc_str:<25} {prec_str:<25} {rec_str:<25} {f1_str:<25}")
    
    print_results_table(standard_results, "Abordagem 1: Regularização Padrão")
    print_results_table(engineered_results, "Abordagem 2: Feature Engineering")
    
    # Analysis de feature selection
    print(f"\n" + "="*60)
    print("FEATURE SELECTION ANALYSIS")
    print("="*60)
    
    for approach_name, results_dict in [("Padrão", standard_results), ("Engineered", engineered_results)]:
        print(f"\n{approach_name.upper()}:")
        for method_key, result in results_dict.items():
            if result['n_features_selected']:
                n_features = result['n_features_selected']
                method_name = result['method']
                print(f"  {method_name}: {np.mean(n_features):.1f} ± {np.std(n_features):.1f} features selected")
    
    # Identify best method
    print(f"\n" + "="*60)
    print("METHODOLOGICAL RECOMMENDATION")
    print("="*60)
    
    all_results = []
    for approach, results_dict in [("Padrão", standard_results), ("Engineered", engineered_results)]:
        for method_key, result in results_dict.items():
            all_results.append({
                'approach': approach,
                'method': result['method'],
                'auc_mean': result['metrics']['auc']['mean'],
                'auc_ci_width': result['metrics']['auc']['ci_upper'] - result['metrics']['auc']['ci_lower'],
                'auc_std': result['metrics']['auc']['std']
            })
    
    # Sort by mean AUC
    all_results.sort(key=lambda x: x['auc_mean'], reverse=True)
    
    print(f"Ranking by mean AUC-ROC:")
    for i, result in enumerate(all_results, 1):
        print(f"{i}. {result['method']} ({result['approach']}): "
              f"AUC = {result['auc_mean']:.3f} ± {result['auc_std']:.3f}")
    
    best_method = all_results[0]
    print(f"\nRECOMMENDED METHOD: {best_method['method']} ({best_method['approach']})")
    print(f"  - AUC-ROC: {best_method['auc_mean']:.3f}")
    print(f"  - Estabilidade (std): {best_method['auc_std']:.3f}")
    
    return {
        'standard_results': standard_results,
        'engineered_results': engineered_results,
        'best_method': best_method,
        'all_results': all_results
    }

# Execute analysis comparativa
if __name__ == "__main__":
    results = comprehensive_regularization_analysis()

COMPARATIVE REGULARIZATION ANALYSIS - MODELO 1
Dataset de treinamento:
  - Observations: 6,588
  - Features: 27
  - Obese: 183 (2.8%)
  - Não-obesos: 6405 (97.2%)

Feature groups identified:
  - Regiao: 5 variables
  - Cor_raca: 4 variables
  - Tipo_parto: 3 variables
  - Demograficas: 3 variables
  - Perinatais: 4 variables
  - Socioeconomicas: 4 variables
  - Outras: 4 variables

APPROACH 1: STANDARD REGULARIZATION

[SEARCH] Evaluating Lasso...

[SEARCH] Evaluating Ridge...

[SEARCH] Evaluating Elastic Net...

APPROACH 2: GROUP-BASED FEATURE ENGINEERING
Features after engineering:
  - Original: 27 features
  - Engineered: 18 features
  - Reduction: 9 features

[SEARCH] Evaluating Lasso (Engineered)...

[SEARCH] Evaluating Ridge (Engineered)...

[SEARCH] Evaluating Elastic Net (Engineered)...

COMPARATIVE RESULTS

APPROACH 1: STANDARD REGULARIZATION:
Método          AUC-ROC                   Precision                 Recall                    F1-Score                 
----------------